In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Fase Gold - Conteudo") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold_projeto LOCATION 'hdfs://hdfs-nn:9000/gold/gold_projeto.db/'")

DataFrame[]

In [4]:
spark.sql("DROP TABLE IF EXISTS gold_projeto.dim_conteudo")

DataFrame[]

In [5]:
spark.sql("""
    CREATE EXTERNAL TABLE gold_projeto.dim_conteudo (
        id_conteudo BIGINT,
        tmdb_id STRING,
        imdb_id STRING,
        title STRING,
        type STRING,
        release_year INT,
        age_certification STRING,
        runtime INT,
        production_countries STRING,
        seasons DOUBLE,
        plat_netflix INT,
        plat_amazon INT,
        genre_scifi INT,
        genre_documentation INT,
        genre_crime INT,
        genre_fantasy INT,
        genre_action INT,
        genre_animation INT,
        genre_sport INT,
        genre_family INT,
        genre_horror INT,
        genre_history INT,
        genre_music INT,
        genre_drama INT,
        genre_western INT,
        genre_war INT,
        genre_european INT,
        genre_romance INT,
        genre_thriller INT,
        genre_reality INT,
        genre_comedy INT
    )
    USING DELTA
    LOCATION 'hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_conteudo/'
""")

DataFrame[]

In [6]:
streaming_titles = spark.table("projeto.streaming_titles")

In [7]:
colunas_para_agrupar = [
    F.col("id").alias("tmdb_id"),
    F.col("imdb_id"),
    F.col("title"),
    F.col("type"),
    F.col("release_year"),
    F.col("age_certification"),
    F.col("runtime"),
    F.col("production_countries"),
    F.col("seasons"),
    F.col("genre_scifi"),
    F.col("genre_documentation"),
    F.col("genre_crime"),
    F.col("genre_fantasy"),
    F.col("genre_action"),
    F.col("genre_animation"),
    F.col("genre_sport"),
    F.col("genre_family"),
    F.col("genre_horror"),
    F.col("genre_history"),
    F.col("genre_music"),
    F.col("genre_drama"),
    F.col("genre_western"),
    F.col("genre_war"),
    F.col("genre_european"),
    F.col("genre_romance"),
    F.col("genre_thriller"),
    F.col("genre_reality"),
    F.col("genre_comedy")
]

In [8]:
df_pivot = streaming_titles.groupBy(colunas_para_agrupar) \
    .pivot("platform") \
    .agg(F.lit(1)) \
    .na.fill(0)

print("Colunas geradas após o pivot:")
print(df_pivot.columns)

Colunas geradas após o pivot:
['tmdb_id', 'imdb_id', 'title', 'type', 'release_year', 'age_certification', 'runtime', 'production_countries', 'seasons', 'genre_scifi', 'genre_documentation', 'genre_crime', 'genre_fantasy', 'genre_action', 'genre_animation', 'genre_sport', 'genre_family', 'genre_horror', 'genre_history', 'genre_music', 'genre_drama', 'genre_western', 'genre_war', 'genre_european', 'genre_romance', 'genre_thriller', 'genre_reality', 'genre_comedy', 'amazon', 'netflix']


In [9]:
novas_colunas = [c for c in df_pivot.columns if c not in [x.alias for x in colunas_para_agrupar] and c not in ["tmdb_id", "imdb_id", "title"]]

In [10]:
for col_name in df_pivot.columns:
    if col_name.lower() in ['netflix', 'amazon']:
        df_pivot = df_pivot.withColumnRenamed(col_name, f"plat_{col_name}")

In [11]:
windowSpec = Window.orderBy("tmdb_id")

df_conteudo_gold = df_pivot \
    .withColumnRenamed("id", "tmdb_id") \
    .withColumn("id_conteudo", F.row_number().over(windowSpec))

In [12]:
print("Verificação do primeiro ID:")
df_conteudo_gold.select("id_conteudo", "title").orderBy("id_conteudo").show(5)

Verificação do primeiro ID:
+-----------+--------------------+
|id_conteudo|               title|
+-----------+--------------------+
|          1|     the lucky texan|
|          2|boonie bears: the...|
|          3|        je suis karl|
|          4|            zone 414|
|          5|              takers|
+-----------+--------------------+
only showing top 5 rows



In [13]:
print("Colunas finais:", df_conteudo_gold.columns)

Colunas finais: ['tmdb_id', 'imdb_id', 'title', 'type', 'release_year', 'age_certification', 'runtime', 'production_countries', 'seasons', 'genre_scifi', 'genre_documentation', 'genre_crime', 'genre_fantasy', 'genre_action', 'genre_animation', 'genre_sport', 'genre_family', 'genre_horror', 'genre_history', 'genre_music', 'genre_drama', 'genre_western', 'genre_war', 'genre_european', 'genre_romance', 'genre_thriller', 'genre_reality', 'genre_comedy', 'plat_amazon', 'plat_netflix', 'id_conteudo']


In [14]:
df_conteudo_gold.toPandas()

,tmdb_id,imdb_id,title,type,release_year,age_certification,runtime,production_countries,seasons,genre_scifi,...,genre_western,genre_war,genre_european,genre_romance,genre_thriller,genre_reality,genre_comedy,plat_amazon,plat_netflix,id_conteudo
0,tm100001,tt0025440,the lucky texan,movie,1934,na,61,[us],0,0,...,1,0,0,1,0,0,0,1,0,1
1,tm1000022,tt11654032,boonie bears: the wild life,movie,2021,na,99,[cn],0,1,...,0,0,0,0,0,0,0,1,0,2
2,tm1000037,tt9205538,je suis karl,movie,2021,r,126,"[cz, de]",0,0,...,0,0,1,1,1,0,0,0,1,3
3,tm1000147,tt8545482,zone 414,movie,2021,r,98,[gb],0,1,...,0,0,0,0,1,0,0,0,1,4
4,tm100015,tt1135084,takers,movie,2010,pg-13,107,[us],0,0,...,0,0,0,0,1,0,0,0,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15620,ts98387,tt6954612,abc galaxy: new space adventures,show,2015,tv-g,13,None,1,0,...,0,0,0,0,0,0,0,1,0,15621
15621,ts98392,tt6953902,technological marvels of the ancient world,show,2005,tv-g,79,None,1,0,...,0,0,0,0,0,0,0,1,0,15622
15622,ts987,tt0081848,danger mouse,show,1981,tv-y,14,[gb],10,1,...,0,0,1,0,0,0,1,0,1,15623
15623,ts9913,tt0938980,crime inc.,show,1984,tv-ma,25,[gb],1,0,...,0,0,1,0,0,0,0,1,0,15624


In [15]:
df_conteudo_gold \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_conteudo")

In [16]:
spark.sql(
    """
    DESCRIBE HISTORY gold_projeto.dim_conteudo
    """
).show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-12-25 11:58:...|  null|    null|       WRITE|{mode -> Overwrit...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 1, n...|        null|Apache-Spark/3.4....|
|      0|2025-12-25 11:58:...|  null|    null|CREATE TABLE|{isManaged -> fal...|null|    null|     null|       null|  Serializable|         true|                  {}|        null|Apache-Spark/3.4.

In [17]:
spark.sql(
    """
    SELECT * FROM gold_projeto.dim_conteudo
    """
).show()

+---------+----------+--------------------+-----+------------+-----------------+-------+--------------------+-------+-----------+-------------------+-----------+-------------+------------+---------------+-----------+------------+------------+-------------+-----------+-----------+-------------+---------+--------------+-------------+--------------+-------------+------------+-----------+------------+-----------+
|  tmdb_id|   imdb_id|               title| type|release_year|age_certification|runtime|production_countries|seasons|genre_scifi|genre_documentation|genre_crime|genre_fantasy|genre_action|genre_animation|genre_sport|genre_family|genre_horror|genre_history|genre_music|genre_drama|genre_western|genre_war|genre_european|genre_romance|genre_thriller|genre_reality|genre_comedy|plat_amazon|plat_netflix|id_conteudo|
+---------+----------+--------------------+-----+------------+-----------------+-------+--------------------+-------+-----------+-------------------+-----------+-------------

In [18]:
spark.table("gold_projeto.dim_conteudo").count()

15625

In [22]:
spark.stop()